In [1]:
# %%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm

import os
import kagglehub

# Step 1: Set up Kaggle credentials (only needed once per environment)
# Make sure you've downloaded kaggle.json from https://www.kaggle.com/account
# and replace 'your_kaggle_username' and 'your_api_key' accordingly,
# or load them from a file or environment variable.

kaggle_username = "djcodesstuff"
kaggle_key = os.environ["KAGGLE_KEY"]

# Set environment variables for Kaggle
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Step 2: Download the LLaMA 3 model from KaggleHub
path = kagglehub.model_download("metaresearch/llama-3/transformers/8b-hf")

print("✅ Download complete.")
print("Path to model files:", path)


/Users/DJ/anaconda3/envs/Spring25_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

Download already complete (4771 bytes).
Download already complete (126 bytes).
Download already complete (7801 bytes).


Download already complete (3276 bytes).
Download already complete (2861 bytes).
Download already complete (1959 bytes).
Download already complete (654 bytes).


Download already complete (46 bytes).
Download already complete (23950 bytes).
Download already complete (433 bytes).
Download already complete (73 bytes).
Download already complete (50941 bytes).
Download already complete (9084490 bytes).
Download already complete (1168138808 bytes).
Resuming download from 4602200064 bytes (397602656 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/models/metaresearch/llama-3/transformers/8b-hf/1/download/model-00002-of-00004.safetensors (4602200064/4999802720) bytes left.


Resuming download from 4818206720 bytes (158491952 bytes left)...


Resuming download from https://www.kaggle.com/api/v1/models/metaresearch/llama-3/transformers/8b-hf/1/download/model-00001-of-00004.safetensors (4818206720/4976698672) bytes left.


Download already complete (4915916176 bytes).




















































100%|██████████| 4.63G/4.63G [00:06<00:00, 24.3MB/s]
100%|██████████| 4.66G/4.66G [00:10<00:00, 37.1MB/s]0:32,  3.58s/it]

✅ Download complete.
Path to model files: /Users/DJ/.cache/kagglehub/models/metaresearch/llama-3/transformers/8b-hf/1


In [ ]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, AutoModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login()  # reads WANDB_API_KEY from the environment  # Replace with your actual API key
wandb.init(project="Spring2025")


In [ ]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

In [ ]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available

    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/content/15_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']



In [ ]:
train_dataset

In [ ]:
# # Load tokenizer
# model_name = "/kaggle/input/llama-3/transformers/8b-hf/1"  # Modify as needed
# # model_name = "meta-llama/Meta-Llama-3-8B"
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenizer.pad_token = tokenizer.eos_token

# # Apply tokenization
# train_dataset = train_dataset.map(tokenize_function, batched=True)
# eval_dataset = eval_dataset.map(tokenize_function, batched=True)

model_path = "/kaggle/input/llama-3/transformers/8b-hf/1"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_skip_modules=None,
    llm_int8_enable_fp32_cpu_offload=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=4096)

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Base model in 4-bit
base_model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="balanced"
)

import torch.nn as nn


# ---------- Custom Classification Head ----------
class LlamaForClassification(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # First token as CLS
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {"loss": loss, "logits": logits}

model = LlamaForClassification(base_model, num_labels=15)

In [ ]:
import random
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ---------- Training Setup ----------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average='weighted', zero_division=0),
        "recall": recall_score(labels, predictions, average='weighted', zero_division=0),
        "f1": f1_score(labels, predictions, average='weighted', zero_division=0)
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    label_names=["labels"],
    logging_steps=10,
    num_train_epochs=15,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    metric_for_best_model="balanced_accuracy",
    load_best_model_at_end=True,
    report_to=["wandb"],
    fp16=False,  # ✅ MUST BE FALSE when using quantized models
    bf16=False,
    max_grad_norm=0.0
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# ---------- Run Training ----------
trainer.train()
results = trainer.evaluate()
print(results)

# Evaluate the model
results = trainer.evaluate()
print(results)

In [ ]:
# 279 labels

In [ ]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available

    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/labels-web-of-law/279_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']




In [ ]:
# # Load tokenizer
# model_name = "/kaggle/input/llama-3/transformers/8b-hf/1"  # Modify as needed
# # model_name = "meta-llama/Meta-Llama-3-8B"
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenizer.pad_token = tokenizer.eos_token

# # Apply tokenization
# train_dataset = train_dataset.map(tokenize_function, batched=True)
# eval_dataset = eval_dataset.map(tokenize_function, batched=True)

model_path = "/kaggle/input/llama-3/transformers/8b-hf/1"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_skip_modules=None,
    llm_int8_enable_fp32_cpu_offload=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Base model in 4-bit
base_model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto"
)

import torch.nn as nn


# ---------- Custom Classification Head ----------
class LlamaForClassification(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # First token as CLS
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {"loss": loss, "logits": logits}

model = LlamaForClassification(base_model, num_labels=15)

In [ ]:
import random
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ---------- Training Setup ----------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average='weighted', zero_division=0),
        "recall": recall_score(labels, predictions, average='weighted', zero_division=0),
        "f1": f1_score(labels, predictions, average='weighted', zero_division=0)
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    label_names=["labels"],
    logging_steps=10,
    num_train_epochs=15,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    metric_for_best_model="balanced_accuracy",
    load_best_model_at_end=True,
    report_to=["wandb"],
    fp16=False,  # ✅ MUST BE FALSE when using quantized models
    bf16=False,
    max_grad_norm=0.0
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# ---------- Run Training ----------
trainer.train()
results = trainer.evaluate()
print(results)

# Evaluate the model
results = trainer.evaluate()
print(results)